In [ ]:
# Adaptive Phase Estimation – Examples

# Cell 1: Imports & seed
import numpy as np
from adaptive_phase_estimation import (
    PhaseEstimationBenchmark,
    export_results,
    simple_noise_model,
    bitstring_to_phase_msb_first,
 )
np.random.seed(42)

# Cell 2: Basic benchmark (exact 5-bit dyadic phase)
bench = PhaseEstimationBenchmark(target_phase=0.65625, precision_bits=5)
results = bench.run_full_benchmark(
    shots_list=[1024, 2048, 4096],
    noise_levels=[0.0, 1e-3, 3e-3],
    replicates=3,
    verbose=True,
 )
bench.plot_benchmark_results(results, save_path="example_results.png")

# Cell 3: Non‑exact (irrational) phase
bench_irr = PhaseEstimationBenchmark(target_phase=np.pi/7, precision_bits=6)
results_irr = bench_irr.run_full_benchmark(
    shots_list=[2048, 4096, 8192],
    noise_levels=[0.0, 1e-3, 5e-3],
    replicates=5,
    verbose=False,
 )
bench_irr.plot_benchmark_results(results_irr, save_path="irrational_results.png")

# Cell 4: Export & quick pandas view
export_results(results_irr, "irrational_benchmark.csv", "irrational_benchmark.json")
import pandas as pd
df = pd.read_csv("irrational_benchmark.csv")
print(df.head())
print("\nMean errors (noisy rows only):")
if "err_static_mean" in df.columns:
    noisy = df[df.regime=="noisy"]
    cols = [c for c in noisy.columns if c.startswith("err_")]
    print(noisy[cols].mean())
else:
    print(df[['err_static','err_dynamic']].mean())

# Cell 5: Precision sweep (single shots set)
for m_bits in [3,4,5,6]:
    b = PhaseEstimationBenchmark(target_phase=1/3, precision_bits=m_bits)
    _ = b.run_full_benchmark(
        shots_list=[4096], noise_levels=[0.0,1e-3], replicates=3, verbose=False
    )
    print(f"Precision {m_bits} done")

# Cell 6: Inspect circuits & basic metrics
b_inspect = PhaseEstimationBenchmark(target_phase=0.375, precision_bits=4)
static_qc = b_inspect.build_static_ipea()
dynamic_qc = b_inspect.build_dynamic_ipea()
print("Static circuit (truncated repr):", static_qc)
print("Dynamic circuit (truncated repr):", dynamic_qc)
from qiskit_aer import AerSimulator
from qiskit import transpile
sim = AerSimulator()
s_t = transpile(static_qc, backend=sim, optimization_level=1)
d_t = transpile(dynamic_qc, backend=sim, optimization_level=1)
print(f"Static: qubits={s_t.num_qubits} depth={s_t.depth()} CX={s_t.count_ops().get('cx',0)}")
print(f"Dynamic: qubits={d_t.num_qubits} depth={d_t.depth()} CX={d_t.count_ops().get('cx',0)}")

# Cell 7: Custom noise quick test
custom_noise = simple_noise_model(p1=0.005, p2=0.025, readout_p=0.05)
b_custom = PhaseEstimationBenchmark(target_phase=0.25, precision_bits=5)
s_qc = b_custom.build_static_ipea()
d_qc = b_custom.build_dynamic_ipea()
s_phase,_ = b_custom.simulate(s_qc, shots=4096, noise=custom_noise, seed=42)
d_phase,_ = b_custom.simulate(d_qc, shots=4096, noise=custom_noise, seed=42)
print(f"Custom noise static err={abs(s_phase-0.25):.4f} dynamic err={abs(d_phase-0.25):.4f}")

# Cell 8: Batch phases (concise)
phases = [0.125,0.25,0.375,0.5,np.pi/4,np.e/10]
batch = []
for ph in phases:
    btmp = PhaseEstimationBenchmark(target_phase=ph, precision_bits=5)
    r = btmp.run_full_benchmark(shots_list=[4096], noise_levels=[0.0,1e-3], replicates=2, verbose=False)
    batch.append((ph,r))
    print(f"Phase {ph:.6f} done")
for i,(ph,r) in enumerate(batch):
    export_results(r, f"batch_{i}_{ph:.4f}.csv", f"batch_{i}_{ph:.4f}.json")

# Cell 9: Helper example – bitstring to phase
for bits in ["1","10","11","101","10101"]:
    print(bits, bitstring_to_phase_msb_first(bits))